In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("Kafka-Streaming") \
    .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages",",".join([ "org.apache.hadoop:hadoop-aws:3.3.4", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"])) \
    .config("spark.hadoop.fs.s3a.endpoint","http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key","admin") \
    .config("spark.hadoop.fs.s3a.secret.key","password123") \
    .config("spark.hadoop.fs.s3a.path.style.access","true") \
    .config("spark.hadoop.fs.s3a.impl","org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=[
        "org.apache.hadoop:hadoop-aws:3.3.4",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"]).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9f5a8236-bd53-40c6-aaea-01f8c72c5f06;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 

In [2]:
# Read Kafka Stream - spark.readStream(Keep listening forever)
stream_df = spark.readStream \
    .format("kafka").option("kafka.bootstrap.servers","kafka:9092") \
    .option("subscribe","user-events") \
    .load()

## Bronze Layer

In [3]:
from pyspark.sql.functions import col

bronze_df = stream_df.select(
    col("value").cast("string").alias("raw_json"),
    col("timestamp")
)

In [4]:
print(bronze_df)

DataFrame[raw_json: string, timestamp: timestamp]


In [5]:
bronze_query = bronze_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "s3a://bronze/checkpoints/user-events"
    ) \
    .start(
        "s3a://bronze/user-events"
    )

26/05/26 16:06:05 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/26 16:06:31 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [6]:
bronze_query.status

{'message': 'Initializing sources',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [7]:
bronze_query.lastProgress

In [8]:
bronze_console_query = bronze_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("truncate", False) \
    .start()

26/05/26 16:06:32 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-02d421fc-7697-4548-b0ba-cd22a6cec8b8. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/26 16:06:32 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## Silver Layer

In [9]:
# Define Schema
from pyspark.sql.types import *

schema = StructType([
    StructField("user_id", IntegerType()),
    StructField("action", StringType()),
    StructField("amount", IntegerType())
])

In [10]:
# Parse Bronze JSON
from pyspark.sql.functions import from_json

silver_df = bronze_df.select(
    from_json(
        col("raw_json"),
        schema
    ).alias("data")
).select("data.*")

In [11]:
silver_df = silver_df.filter(
    col("amount") > 0
)

In [12]:
silver_query = silver_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option(
        "checkpointLocation",
        "s3a://silver/checkpoints/user-events"
    ) \
    .start(
        "s3a://silver/user-events"
    )

26/05/26 16:06:33 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/05/26 16:06:33 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+---------+
|raw_json|timestamp|
+--------+---------+
+--------+---------+



26/05/26 16:06:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------------+-----------------------+
|raw_json                                        |timestamp              |
+------------------------------------------------+-----------------------+
|{"user_id": 6, "action": "login", "amount": 158}|2026-05-26 16:06:35.143|
+------------------------------------------------+-----------------------+



26/05/26 16:06:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [13]:
silver_console_query = silver_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("truncate", False) \
    .start()

26/05/26 16:06:40 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-780d57be-813b-4cdd-81b8-aa242ad2be26. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/26 16:06:40 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## Gold Layer: Business metrics

In [14]:
gold_df = silver_df.groupBy(
    "action"
).count()

26/05/26 16:06:40 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/05/26 16:06:40 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [15]:
gold_query = gold_df.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option(
        "checkpointLocation",
        "s3a://gold/checkpoints/action-counts"
    ) \
    .start(
        "s3a://gold/action-counts"
    )

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------------------+-----------------------+
|raw_json                                              |timestamp              |
+------------------------------------------------------+-----------------------+
|{"user_id": 30, "action": "purchase", "amount": 152}  |2026-05-26 16:06:36.145|
|{"user_id": 57, "action": "login", "amount": 195}     |2026-05-26 16:06:37.148|
|{"user_id": 17, "action": "purchase", "amount": 57}   |2026-05-26 16:06:38.15 |
|{"user_id": 7, "action": "add_to_cart", "amount": 364}|2026-05-26 16:06:39.152|
|{"user_id": 25, "action": "login", "amount": 142}     |2026-05-26 16:06:40.154|
+------------------------------------------------------+-----------------------+

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+------+------+
|user_id|action|amount|
+-------+------+------+
+---

26/05/26 16:06:42 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 3
-------------------------------------------
+----------------------------------------------------+-----------------------+
|raw_json                                            |timestamp              |
+----------------------------------------------------+-----------------------+
|{"user_id": 97, "action": "purchase", "amount": 133}|2026-05-26 16:06:41.156|
+----------------------------------------------------+-----------------------+



In [17]:
bronze_query.status

{'message': 'Processing new data',
 'isDataAvailable': True,
 'isTriggerActive': True}

-------------------------------------------
Batch: 30
-------------------------------------------
+-------------------------------------------------------+-----------------------+
|raw_json                                               |timestamp              |
+-------------------------------------------------------+-----------------------+
|{"user_id": 40, "action": "add_to_cart", "amount": 160}|2026-05-26 16:07:50.873|
+-------------------------------------------------------+-----------------------+

-------------------------------------------
Batch: 18
-------------------------------------------
+-------+-----------+------+
|user_id|action     |amount|
+-------+-----------+------+
|24     |logout     |32    |
|24     |logout     |412   |
|40     |add_to_cart|160   |
+-------+-----------+------+



[Stage 101:>                (0 + 1) / 1][Stage 107:>                (0 + 1) / 1]

-------------------------------------------
Batch: 31
-------------------------------------------
+---------------------------------------------------+-----------------------+
|raw_json                                           |timestamp              |
+---------------------------------------------------+-----------------------+
|{"user_id": 68, "action": "login", "amount": 87}   |2026-05-26 16:07:51.88 |
|{"user_id": 90, "action": "purchase", "amount": 63}|2026-05-26 16:07:52.885|
+---------------------------------------------------+-----------------------+



-------------------------------------------
Batch: 19
-------------------------------------------
-------------------------------------------
Batch: 32
-------------------------------------------
+-------+--------+------+
|user_id|action  |amount|
+-------+--------+------+
|68     |login   |87    |
|90     |purchase|63    |
+-------+--------+------+

+-------------------------------------------------------+-----------------------+
|raw_json                                               |timestamp              |
+-------------------------------------------------------+-----------------------+
|{"user_id": 71, "action": "add_to_cart", "amount": 424}|2026-05-26 16:07:53.886|
+-------------------------------------------------------+-----------------------+

-------------------------------------------
Batch: 33
-------------------------------------------
Batch: 20
-------------------------------------------
-------------------------------------------
+---------------------------------------

-------------------------------------------
Batch: 35
-------------------------------------------
-------------------------------------------
Batch: 22
-------------------------------------------
+------------------------------------------------------+-----------------------+
|raw_json                                              |timestamp              |
+------------------------------------------------------+-----------------------+
|{"user_id": 94, "action": "add_to_cart", "amount": 64}|2026-05-26 16:07:56.893|
+------------------------------------------------------+-----------------------+

+-------+-----------+------+
|user_id|action     |amount|
+-------+-----------+------+
|94     |add_to_cart|64    |
+-------+-----------+------+



[Stage 121:(52 + 24) / 200][Stage 124:>  (0 + 0) / 5][Stage 125:>  (0 + 0) / 2]]

In [18]:
silver_query.status

{'message': 'Processing new data',
 'isDataAvailable': True,
 'isTriggerActive': True}

[Stage 121:(106 + 24) / 200][Stage 124:>  (0 + 0) / 5][Stage 125:>  (0 + 0) / 2]

In [21]:
gold_query.status

{'message': 'Processing new data',
 'isDataAvailable': True,
 'isTriggerActive': True}

-------------------------------------------
Batch: 52
-------------------------------------------
+-------+------+------+
|user_id|action|amount|
+-------+------+------+
|68     |logout|47    |
+-------+------+------+

-------------------------------------------
Batch: 65
-------------------------------------------
+-------------------------------------------------+-----------------------+
|raw_json                                         |timestamp              |
+-------------------------------------------------+-----------------------+
|{"user_id": 68, "action": "logout", "amount": 47}|2026-05-26 16:09:07.162|
+-------------------------------------------------+-----------------------+

-------------------------------------------
Batch: 53
-------------------------------------------
+-------+-----------+------+
|user_id|action     |amount|
+-------+-----------+------+
|87     |logout     |283   |
|71     |purchase   |64    |
|13     |purchase   |375   |
|92     |purchase   |245   |
|

-------------------------------------------
Batch: 66
-------------------------------------------
+-------------------------------------------------------+-----------------------+
|raw_json                                               |timestamp              |
+-------------------------------------------------------+-----------------------+
|{"user_id": 87, "action": "logout", "amount": 283}     |2026-05-26 16:09:08.164|
|{"user_id": 71, "action": "purchase", "amount": 64}    |2026-05-26 16:09:09.166|
|{"user_id": 13, "action": "purchase", "amount": 375}   |2026-05-26 16:09:10.172|
|{"user_id": 92, "action": "purchase", "amount": 245}   |2026-05-26 16:09:11.173|
|{"user_id": 49, "action": "add_to_cart", "amount": 342}|2026-05-26 16:09:12.179|
|{"user_id": 41, "action": "purchase", "amount": 240}   |2026-05-26 16:09:13.181|
|{"user_id": 9, "action": "purchase", "amount": 367}    |2026-05-26 16:09:14.183|
|{"user_id": 72, "action": "login", "amount": 233}      |2026-05-26 16:09:15.188|


-------------------------------------------
Batch: 55
-------------------------------------------
+-------+------+------+
|user_id|action|amount|
+-------+------+------+
|69     |logout|106   |
+-------+------+------+

-------------------------------------------
Batch: 67
-------------------------------------------
+--------------------------------------------------+-----------------------+
|raw_json                                          |timestamp              |
+--------------------------------------------------+-----------------------+
|{"user_id": 69, "action": "logout", "amount": 106}|2026-05-26 16:09:16.654|
+--------------------------------------------------+-----------------------+



-------------------------------------------
Batch: 68
-------------------------------------------
+-------------------------------------------------------+-----------------------+
|raw_json                                               |timestamp              |
+-------------------------------------------------------+-----------------------+
|{"user_id": 67, "action": "add_to_cart", "amount": 11} |2026-05-26 16:09:17.657|
|{"user_id": 53, "action": "add_to_cart", "amount": 289}|2026-05-26 16:09:18.662|
+-------------------------------------------------------+-----------------------+

-------------------------------------------
Batch: 56
-------------------------------------------
+-------+-----------+------+
|user_id|action     |amount|
+-------+-----------+------+
|67     |add_to_cart|11    |
|53     |add_to_cart|289   |
+-------+-----------+------+

-------------------------------------------
Batch: 69
-------------------------------------------
+-------------------------------------

...

-------------------------------------------
Batch: 116
-------------------------------------------
+-------+--------+------+
|user_id|action  |amount|
+-------+--------+------+
|78     |purchase|283   |
+-------+--------+------+

-------------------------------------------
Batch: 128
-------------------------------------------
+----------------------------------------------------+-----------------------+
|raw_json                                            |timestamp              |
+----------------------------------------------------+-----------------------+
|{"user_id": 78, "action": "purchase", "amount": 283}|2026-05-26 16:11:15.552|
+----------------------------------------------------+-----------------------+

-------------------------------------------
Batch: 117
-------------------------------------------
+-------+--------+------+
|user_id|action  |amount|
+-------+--------+------+
|31     |purchase|477   |
|49     |login   |43    |
+-------+--------+------+

-------------------

[Stage 635:>                                                        (0 + 1) / 1]

-------------------------------------------
Batch: 130
-------------------------------------------
+----------------------------------------------------+-----------------------+
|raw_json                                            |timestamp              |
+----------------------------------------------------+-----------------------+
|{"user_id": 91, "action": "purchase", "amount": 472}|2026-05-26 16:11:18.568|
+----------------------------------------------------+-----------------------+

-------------------------------------------
Batch: 118
-------------------------------------------
+-------+--------+------+
|user_id|action  |amount|
+-------+--------+------+
|91     |purchase|472   |
+-------+--------+------+



-------------------------------------------
Batch: 131
-------------------------------------------
-------------------------------------------
Batch: 119
-------------------------------------------
+-------+--------+------+
|user_id|action  |amount|
+-------+--------+------+
|64     |purchase|238   |
+-------+--------+------+

+----------------------------------------------------+-----------------------+
|raw_json                                            |timestamp              |
+----------------------------------------------------+-----------------------+
|{"user_id": 64, "action": "purchase", "amount": 238}|2026-05-26 16:11:19.573|
+----------------------------------------------------+-----------------------+

-------------------------------------------
Batch: 132
-------------------------------------------
-------------------------------------------
Batch: 120
-------------------------------------------
+-------------------------------------------------------+--------------------

-------------------------------------------
Batch: 122
-------------------------------------------
-------------------------------------------
Batch: 134
-------------------------------------------
+-------+--------+------+
|user_id|action  |amount|
+-------+--------+------+
|59     |purchase|401   |
+-------+--------+------+

+----------------------------------------------------+-----------------------+
|raw_json                                            |timestamp              |
+----------------------------------------------------+-----------------------+
|{"user_id": 59, "action": "purchase", "amount": 401}|2026-05-26 16:11:30.061|
+----------------------------------------------------+-----------------------+

